# Western-U.S. quick threshold transfer

A six-lake, NDWI-only transfer experiment using bounded Earth Engine samples and entirely local threshold fitting. It creates no Earth Engine export tasks.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'configs' / 'western_us.yaml').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'configs' / 'western_us.yaml').exists():
    raise FileNotFoundError('Run from the repository or notebooks directory.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from regional_analysis.config import load_config
from regional_analysis.discover_lakes import initialize_earth_engine
from regional_analysis.quick_examples import QUICK_EXAMPLE_SETS
from regional_analysis.quick_sampling import (
    QuickSamplingSettings, load_candidate_inventory, select_lake_specs
)
from regional_analysis.quick_transfer import (
    load_transfer_tables, run_transfer_experiment, write_transfer_results
)

REFERENCE = 'OPERA'  # switch to JRC only after OPERA completes
RUN_LIVE = True  # switch to False after compact results have been saved
OUTPUT_DIR = REPO_ROOT / 'data' / 'regional'

## 2. Run once or load compact results

Each live run retrieves every lake exactly once for the selected reference. Sample rows remain in memory and are never written.

In [ ]:
if RUN_LIVE:
    config = load_config(REPO_ROOT / 'configs' / 'western_us.yaml')
    inventory = load_candidate_inventory(
        OUTPUT_DIR / 'western_hydrolakes_candidates.csv'
    )
    lakes = select_lake_specs(inventory, QUICK_EXAMPLE_SETS['saline6'])
    settings = QuickSamplingSettings(
        buffer_km=5, samples_per_image=20,
        analysis_end=pd.Timestamp('2026-08-09').date(),
    )
    ee, project = initialize_earth_engine()
    print(f'Earth Engine project: {project}')
    result = run_transfer_experiment(
        ee, config.earth_engine.hydrolakes_asset, lakes, REFERENCE, settings
    )
    write_transfer_results(result, OUTPUT_DIR)
    tables = {
        'local': result.local_summary,
        'cross_iou': result.cross_iou,
        'cross_f1': result.cross_f1,
        'regional': result.regional,
        'lolo': result.lolo,
    }
else:
    tables = load_transfer_tables(OUTPUT_DIR, REFERENCE)

## 3. Local spatial-CV benchmarks

In [ ]:
local = tables['local'].copy()
display(local[[
    'Hylak_id', 'Lake_name', 'n_samples', 'sample_retrieval_seconds',
    'local_threshold', 'local_cv_iou', 'local_test_iou', 'local_test_f1',
]])

## 4. Source-threshold × target-lake matrices

Rows identify the lake supplying the local threshold; columns identify the lake supplying untouched test blocks.

In [ ]:
lake_names = local.set_index('Hylak_id')['Lake_name'].to_dict()
cross_iou = tables['cross_iou'].copy()
cross_f1 = tables['cross_f1'].copy()
cross_iou.index = [lake_names[int(value)] for value in cross_iou.index]
cross_iou.columns = [lake_names[int(value)] for value in cross_iou.columns]
cross_f1.index = [lake_names[int(value)] for value in cross_f1.index]
cross_f1.columns = [lake_names[int(value)] for value in cross_f1.columns]
display(cross_iou.style.format('{:.3f}').background_gradient(cmap='YlGn'))
display(cross_f1.style.format('{:.3f}').background_gradient(cmap='YlGn'))

## 5. Equal-lake-weighted pooled regional threshold

In [ ]:
regional = tables['regional'].copy()
display(regional[[
    'Hylak_id', 'Lake_name', 'regional_threshold', 'macro_training_iou',
    'regional_test_iou', 'regional_test_f1', 'macro_test_iou',
    'local_test_iou', 'iou_delta_vs_local',
]])

## 6. Leave-one-lake-out transfer

In [ ]:
lolo = tables['lolo'].copy()
display(lolo[[
    'held_out_Hylak_id', 'held_out_lake', 'threshold',
    'held_out_test_iou', 'held_out_test_f1',
    'held_out_test_precision', 'held_out_test_recall',
    'local_threshold', 'local_test_iou', 'iou_delta_vs_local',
]])
print(f"Macro LOLO IoU: {lolo['macro_lolo_iou'].iloc[0]:.3f}")

## 7. Notes

The pooled and leave-one-lake-out fits maximize the unweighted mean of lake-level training IoUs, so lakes with more sampled rows do not dominate. Each fold derives its deterministic 150-value threshold grid only from eligible training lakes, using the union of their per-lake 0.5th–99.5th percentile ranges. Held-out lakes contribute only untouched test blocks to final scoring. These remain exploratory six-lake results, not final regional thresholds; Dynamic World and additional lakes are outside this task.